In [ ]:
# %%
# Imports
import os
import numpy as np
import pandas as pd 
from scipy.stats import linregress
from tqdm import tqdm


# %%
def calculate_elo(
    teams, data, initial_rating=2000, k=140, width=None, alpha=None, weights=False, lowerlim=float("-inf")
    ):
    '''
    Calculate Elo ratings for each team based on historical match data.

    Parameters:
    - teams (array-like): Array of unique Team IDs.
    - data (pd.DataFrame): Match results sorted in chronological order.
      Must contain columns: WTeamID, LTeamID, WScore, LScore, weight.
    - initial_rating (float): Starting Elo rating for all teams (default: 2000).
    - k (float): K-factor controlling how much each game affects ratings (default: 140).
    - width (float or None): Scaling factor for the logistic function. Defaults to initial_rating if None.
    - alpha (float or None): Divisor for margin-of-victory multiplier. Disabled if None.
    - weights (bool): Unused parameter retained for interface compatibility.
    - lowerlim (float): Floor value for Elo ratings (default: -inf, i.e., no floor).

    Returns:
    - r1 (list[float]): Post-game Elo ratings of the winning team for each match.
    - r2 (list[float]): Post-game Elo ratings of the losing team for each match.
    - loss (list[float]): Brier score component (1 - expected_win_prob)^2 for each match.
    '''
    
    # Initialize all teams with the same starting rating
    team_dict = {}
    for team in teams:
        team_dict[team] = initial_rating
    if not width:
        width = initial_rating
    
    # Storage for per-game results
    r1, r2 = [], []
    loss = []
    margin_of_victory = 1

    # Process each game sequentially to update Elo ratings
    for wteam, lteam, ws, ls, w  in tqdm(
        zip(data.WTeamID, data.LTeamID, data.WScore, data.LScore, data.weight), total=len(data)
    ):

        # Expected win probability for each side using the logistic Elo formula
        rateW = 1 / (1 + 10 ** ((team_dict[lteam] - team_dict[wteam]) / width))
        rateL = 1 / (1 + 10 ** ((team_dict[wteam] - team_dict[lteam]) / width))
        
        # Scale rating update by score difference if alpha is provided
        if alpha:
            margin_of_victory = (ws - ls)/alpha

        # Apply Elo update: winner gains, loser loses (weighted by k, margin, and game weight)
        team_dict[wteam] += w * k * margin_of_victory * (1 - rateW)
        team_dict[lteam] += w * k * margin_of_victory * (0 - rateL)

        # Clamp the loser's rating to the specified floor
        if team_dict[lteam] < lowerlim:
            team_dict[lteam] = lowerlim
            
        # Record post-game ratings and Brier score component
        r1.append(team_dict[wteam])
        r2.append(team_dict[lteam])
        loss.append((1-rateW)**2)
        
    return r1, r2, loss

def create_elo_data(
    teams, data, initial_rating=2000, k=140, width=None, alpha=None, weights=None, lowerlim=float("-inf")
    ):
    '''
    Run the Elo algorithm and produce per-team, per-season summary statistics.

    Parameters:
    - teams (array-like): Array of unique Team IDs.
    - data (pd.DataFrame): Match results sorted in chronological order.
    - initial_rating (float): Starting Elo rating for all teams (default: 2000).
    - k (float): K-factor controlling rating volatility (default: 140).
    - width (float or None): Logistic scaling factor. Defaults to initial_rating if None.
    - alpha (float or None): Margin-of-victory divisor. Disabled if None.
    - weights (array-like or None): Per-game weights applied to Elo updates. Uniform if None.
    - lowerlim (float): Floor value for Elo ratings (default: -inf).

    Returns:
    - results (pd.DataFrame): Per-team, per-season aggregated Elo statistics
      (mean, median, std, min, max, last, trend) using regular-season games only.
    - loss (float): Mean Brier score computed exclusively on tournament games.
    '''
    
    # Assign per-game weights; default to uniform weighting
    if isinstance(weights, (list, np.ndarray, pd.Series)):
        data['weight'] = weights
    else:
        data['weight'] = 1
    
    r1, r2, loss = calculate_elo(
        teams, data, initial_rating, k, width, alpha, weights, lowerlim
    )
    # Evaluate prediction accuracy on tournament games only
    loss = np.mean(np.array(loss)[data.tourney == 1])
    print(f"=== Brier Score: {loss:.5f} (Only  Tournaments) ===")
    
    # Stack winner and loser records into a single long-format DataFrame
    seasons = np.concatenate([data.Season, data.Season])
    days = np.concatenate([data.DayNum, data.DayNum])
    teams = np.concatenate([data.WTeamID, data.LTeamID])
    tourney = np.concatenate([data.tourney, data.tourney])
    ratings = np.concatenate([r1, r2])
    rating_df = pd.DataFrame({
        'Season': seasons,
        'DayNum': days,
        'TeamID': teams,
        'Rating': ratings,
        'Tourney': tourney
    })

    # Keep only regular-season ratings for summary statistics
    rating_df.sort_values(['TeamID', 'Season', 'DayNum'], inplace=True)
    rating_df = rating_df[rating_df['Tourney'] == 0]

    # Aggregate per-team, per-season Elo statistics
    grouped = rating_df.groupby(['TeamID', 'Season'])
    results = grouped['Rating'].agg(['mean', 'median', 'std', 'min', 'max', 'last'])
    results.columns = ['Rating_Mean', 'Rating_Median', 'Rating_Std', 'Rating_Min', 'Rating_Max', 'Rating_Last']

    # Compute linear trend (slope) of Elo across the season
    results['Rating_Trend'] = grouped.apply(lambda x: linregress(range(len(x)), x['Rating']).slope, include_groups=False)
    results.reset_index(inplace=True)
    
    return results, loss

# %% [markdown]
# # Apply Functions and Save Results to CSV

# %%
# --- Men's Tournament Data ---
regular_m = pd.read_csv('/kaggle/input/datasets/deepak0003/march-machine-learning-mania-2026-dataset/MRegularSeasonCompactResults.csv')
tourney_m = pd.read_csv('/kaggle/input/datasets/deepak0003/march-machine-learning-mania-2026-dataset/MNCAATourneyCompactResults.csv')
teams_m = pd.read_csv('/kaggle/input/datasets/deepak0003/march-machine-learning-mania-2026-dataset/MTeams.csv')

# Tag game type and assign weights (regular season weighted higher than tourney)
regular_m['tourney'] = 0
tourney_m['tourney'] = 1
regular_m['weight'] = 1
tourney_m['weight'] = 0.75

# Merge and sort chronologically
data_m = pd.concat([regular_m, tourney_m])
data_m.sort_values(['Season', 'DayNum'], inplace=True)
data_m.reset_index(inplace=True, drop=True)

initial_rating = width = 1200 

print("Men's Ratings")

elo_df_men, _ = create_elo_data(
    teams_m.TeamID, data_m, initial_rating=initial_rating, k=125, width=width, alpha=None, weights=data_m['weight']
)

# %%
# --- Women's Tournament Data ---
regular_w = pd.read_csv('/kaggle/input/datasets/deepak0003/march-machine-learning-mania-2026-dataset/WRegularSeasonCompactResults.csv')
tourney_w = pd.read_csv('/kaggle/input/datasets/deepak0003/march-machine-learning-mania-2026-dataset/WNCAATourneyCompactResults.csv')
teams_w = pd.read_csv('/kaggle/input/datasets/deepak0003/march-machine-learning-mania-2026-dataset/WTeams.csv')

# Tag game type and assign weights (tourney weighted higher for women)
regular_w['tourney'] = 0
tourney_w['tourney'] = 1
regular_w['weight'] = 0.95
tourney_w['weight'] = 1

# Merge and sort chronologically
data_w = pd.concat([regular_w, tourney_w])
data_w.sort_values(['Season', 'DayNum'], inplace=True)
data_w.reset_index(inplace=True, drop=True)

print("Women's Ratings")

elo_df_women, _ = create_elo_data(
    teams_w.TeamID, data_w, initial_rating=initial_rating, k=190, width=width, alpha=None, weights=data_w['weight']
)

# %%
# --- Generate Submission File ---
submission = pd.read_csv("/kaggle/input/datasets/deepak0003/march-machine-learning-mania-2026-dataset/SampleSubmissionStage2.csv")

# Parse matchup IDs into season and team ID columns
sub = submission.ID.str.split('_', expand=True).astype(int)
sub.columns = ["Season", "T1_TeamID", "T2_TeamID"]

# Build a lookup dict of 2026-season final Elo ratings for all teams (men + women)
elo_dict = pd.concat([
    elo_df_women[elo_df_women.Season == 2026],
    elo_df_men[elo_df_men.Season == 2026]
]).set_index("TeamID")["Rating_Last"].to_dict()

# Predict win probability for T1 using the Elo logistic formula
submission.Pred = 1 / (1 + 10**((sub.T2_TeamID.map(elo_dict) - sub.T1_TeamID.map(elo_dict))/width))

submission.to_csv("submission.csv", index=False)

# --- Post-processing: Override predictions for known champions ---
updated_df = pd.read_csv('submission.csv', index_col=0)

new_updated_df = updated_df.copy()

man_1st = 1276    # Predicted men's champion team ID
woman_1st = 3417  # Predicted women's champion team ID

# Force 100% / 0% predictions for matchups involving the predicted champions
for i in range(len(new_updated_df)):
    id_index = new_updated_df.index[i]
    id_str = str(id_index)
    year, team1, team2 = map(int, id_str.split('_'))
    if man_1st == team1 or woman_1st == team1:
        new_updated_df.at[id_index, 'Pred'] = 1
    elif man_1st == team2 or woman_1st == team2:
        new_updated_df.at[id_index, 'Pred'] = 0

new_updated_df.to_csv('submission.csv', index=True)